## 0 · Configuration & Imports

In [0]:
import re
import json
import logging
from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, IntegerType, DateType, TimestampType

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("silver_layer")

spark = SparkSession.builder.appName("youtube_silver").getOrCreate()

## 1 · Pipeline Parameters

In [0]:
dbutils.widgets.text("bucket_name",  "youtube-capstone-bucket-team6", "S3 Bucket Name")
dbutils.widgets.text("catalog_name", "youtube-capstone-catalog",     "Unity Catalog Name")
dbutils.widgets.text("run_date",     datetime.now(timezone.utc).strftime("%Y-%m-%d"), "Run Date")

BUCKET_NAME  = dbutils.widgets.get("bucket_name")
CATALOG_NAME = dbutils.widgets.get("catalog_name")
RUN_DATE     = dbutils.widgets.get("run_date")

BASE_PATH       = f"s3://{BUCKET_NAME}/youtube_pipeline"
BRONZE_PATH     = f"{BASE_PATH}/bronze"
SILVER_PATH     = f"{BASE_PATH}/silver"

BRONZE_DB = f"{CATALOG_NAME}.bronze"
SILVER_DB = f"{CATALOG_NAME}.silver"

logger.info(f"Silver config → catalog={CATALOG_NAME}, run_date={RUN_DATE}")

2026-04-27 08:19:02,925 [INFO] Received command c on object id p0
2026-04-27 08:19:02,972 [INFO] Silver config → catalog=youtube-capstone-catalog, run_date=2026-04-19


## 2 · Helper / Transformation Functions

In [0]:


def deduplicate(df: DataFrame, pk_columns: list[str], order_col: str) -> DataFrame:
    
    from pyspark.sql.window import Window

    window_spec = (
        Window
        .partitionBy(*pk_columns)
        .orderBy(F.col(order_col).desc())
    )

    return (
        df
        .withColumn("_row_rank", F.row_number().over(window_spec))
        .filter(F.col("_row_rank") == 1)
        .drop("_row_rank")
    )

@F.udf(returnType=LongType())
def parse_iso_duration_seconds(duration_str: str) -> int:
    if not duration_str:
        return 0
    pattern = r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?"
    match = re.match(pattern, duration_str)
    if not match:
        return 0
    hours   = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)
    return hours * 3600 + minutes * 60 + seconds




@F.udf(returnType=F.StringType())
def clean_tags_udf(tags_json: str) -> str:
    """
    Parse JSON tag array → lowercase, deduplicated, comma-separated string.
    Returns empty string on parse failure.
    """
    if not tags_json:
        return ""
    try:
        tags = json.loads(tags_json)
        cleaned = list(dict.fromkeys(t.strip().lower() for t in tags if t.strip()))
        return ",".join(cleaned)
    except (json.JSONDecodeError, TypeError):
        return ""


def standardize_text_columns(df: DataFrame, cols: list[str]) -> DataFrame:
    """Trim whitespace and apply consistent casing to free-text fields."""
    for col_name in cols:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
    return df


def fill_numeric_nulls(df: DataFrame, cols: list[str], fill_value: int = 0) -> DataFrame:
    """Replace NULL numeric values with a safe default."""
    fill_map = {c: fill_value for c in cols}
    return df.fillna(fill_map)


def add_silver_metadata(df: DataFrame) -> DataFrame:
    """Stamp processed_at and layer identifier."""
    return (
        df
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("_data_layer",          F.lit("silver"))
    )

2026-04-27 08:19:06,320 [INFO] Received command c on object id p0


In [0]:
logger.info("── Building silver_videos ──")

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{CATALOG_NAME}`.silver")

df_bronze_videos = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/bronze_videos")
)

df_bronze_videos = deduplicate(
    df         = df_bronze_videos,
    pk_columns = ["video_id", "run_date"],
    order_col  = "ingested_at",
)

logger.info(f"[Dedup] bronze_videos → {df_bronze_videos.count():,} unique video_id rows after deduplication")

drop_cols = [c for c in df_bronze_videos.columns if c.startswith("_source")]
df_v = df_bronze_videos.drop(*drop_cols)

df_v = fill_numeric_nulls(df_v, ["view_count", "like_count", "comment_count"])
df_v = df_v.fillna({
    "title":         "Unknown Title",
    "channel_title": "Unknown Channel",
    "category_id":   "0",
    "region_code":   "UNKNOWN",
    "description":   "",
    "tags":          "[]",
    "duration":      "PT0S",
})

# 3d. Text standardisation
df_v = standardize_text_columns(df_v, ["title", "channel_title", "description"])

# 3e. Date / time parsing
df_v = (
    df_v
    .withColumn("published_at",  F.to_timestamp(F.col("publish_date"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("publish_date",  F.to_date(F.col("publish_date"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("publish_year",  F.year(F.col("publish_date")))
    .withColumn("publish_month", F.month(F.col("publish_date")))
    .withColumn("publish_dow",   F.dayofweek(F.col("publish_date")))  # 1=Sunday … 7=Saturday
)

# 3f. Duration enrichment
df_v = df_v.withColumn("duration_seconds", parse_iso_duration_seconds(F.col("duration")))

# 3g. Tags cleanup
df_v = df_v.withColumn("tags_clean", clean_tags_udf(F.col("tags"))).drop("tags")

# 3h. Derived engagement columns (light; deep metrics live in Gold)
df_v = (
    df_v
    .withColumn("views_k",  (F.col("view_count")    / 1000).cast("double"))
    .withColumn("likes_k",  (F.col("like_count")    / 1000).cast("double"))
)


df_v = (
    df_v
    .filter(F.col("video_id").isNotNull() & (F.col("video_id") != ""))
)

df_v = add_silver_metadata(df_v)

(
    df_v
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .partitionBy("region_code", "publish_year")
    .save(f"{SILVER_PATH}/silver_videos")
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS `{CATALOG_NAME}`.silver.silver_videos
    USING DELTA
    LOCATION '{SILVER_PATH}/silver_videos'
""")
spark.sql(f"OPTIMIZE `{CATALOG_NAME}`.silver.silver_videos ZORDER BY (video_id, run_date)")

sv_count = spark.read.format("delta").load(f"{SILVER_PATH}/silver_videos").count()
logger.info(f"silver_videos written: {sv_count:,} rows")

2026-04-27 08:21:05,373 [INFO] Received command c on object id p0
2026-04-27 08:21:05,391 [INFO] ── Building silver_videos ────────────────────────────────────────────
2026-04-27 08:21:06,614 [INFO] [Dedup] bronze_videos → 668 unique video_id rows after deduplication
2026-04-27 08:21:12,815 [INFO] silver_videos written: 668 rows


## 4 · Silver Comments

In [0]:
logger.info("── Building silver_comments ──────────────────────────────────────────")

df_bronze_comments = (
    spark.read
    .format("delta")
    .load(f"{BRONZE_PATH}/bronze_comments")
)
df_bronze_comments = deduplicate(
    df         = df_bronze_comments,
    pk_columns = ["comment_id","run_date"],
    order_col  = "ingested_at",
)

drop_cols = [c for c in df_bronze_comments.columns if c.startswith("_source")]
df_c = df_bronze_comments.drop(*drop_cols)

# 4b. Null handling
df_c = fill_numeric_nulls(df_c, ["like_count"])
df_c = df_c.fillna({
    "author":       "Anonymous",
    "comment_text": "",
})

# 4c. Text cleaning
df_c = standardize_text_columns(df_c, ["author", "comment_text"])

# 4d. Remove empty / bot-like comments
df_c = df_c.filter(
    F.col("comment_id").isNotNull()
    & F.col("video_id").isNotNull()
    & (F.length(F.trim(F.col("comment_text"))) > 0)
)

# 4e. Date parsing
df_c = (
    df_c
    .withColumn("comment_ts",    F.to_timestamp(F.col("comment_date"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("comment_date",  F.to_date(F.col("comment_date"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("comment_year",  F.year(F.col("comment_date")))
    .withColumn("comment_month", F.month(F.col("comment_date")))
)

df_c = (
    df_c
    .withColumn("word_count",    F.size(F.split(F.trim(F.col("comment_text")), r"\s+")))
    .withColumn("char_count",    F.length(F.col("comment_text")))
    .withColumn("has_emoji",     F.col("comment_text").rlike(
        u"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF\U0001F700-\U0001F77F]"
    ))
)

df_c = df_c.dropDuplicates(["comment_id"])

df_c = add_silver_metadata(df_c)

(
    df_c
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .partitionBy("comment_year", "comment_month")
    .save(f"{SILVER_PATH}/silver_comments")
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS `{CATALOG_NAME}`.silver.silver_comments
    USING DELTA
    LOCATION '{SILVER_PATH}/silver_comments'
""")
spark.sql(f"OPTIMIZE `{CATALOG_NAME}`.silver.silver_comments ZORDER BY (video_id, comment_date)")

sc_count = spark.read.format("delta").load(f"{SILVER_PATH}/silver_comments").count()
logger.info(f"silver_comments written: {sc_count:,} rows")

2026-04-27 08:21:57,615 [INFO] Received command c on object id p0
2026-04-27 08:21:57,635 [INFO] ── Building silver_comments ──────────────────────────────────────────
2026-04-27 08:22:06,178 [INFO] silver_comments written: 22,915 rows


## 5 · Checkpoint — Record Latest Processed Run Date

In [0]:
checkpoint_data = {
    "layer":       "silver",
    "last_run_date": RUN_DATE,
    "processed_at":  datetime.now(timezone.utc).isoformat(),
    "silver_videos_count":   sv_count,
    "silver_comments_count": sc_count,
}



2026-04-27 08:22:21,371 [INFO] Received command c on object id p0


## 6 · Silver Data Quality Checks

In [0]:
def dq_silver(path: str, name: str, checks: dict) -> None:
    """
    Run simple column-level DQ assertions.
    checks = {col_name: ("not_null" | "positive")}
    """
    df = spark.read.format("delta").load(path)
    total = df.count()
    logger.info(f"[DQ-Silver] {name}: {total:,} rows")

    for col_name, rule in checks.items():
        if rule == "not_null":
            fails = df.filter(F.col(col_name).isNull()).count()
        elif rule == "positive":
            fails = df.filter(F.col(col_name) < 0).count()
        else:
            fails = 0
        status = "✓" if fails == 0 else f"⚠ {fails:,} violations"
        logger.info(f"  [{rule}] {col_name}: {status}")

dq_silver(f"{SILVER_PATH}/silver_videos",   "silver_videos",   {
    "video_id":         "not_null",
    "view_count":       "positive",
    "like_count":       "positive",
    "duration_seconds": "positive",
    "publish_date":     "not_null",
})

dq_silver(f"{SILVER_PATH}/silver_comments", "silver_comments", {
    "comment_id":   "not_null",
    "video_id":     "not_null",
    "comment_text": "not_null",
})

2026-04-27 08:22:27,099 [INFO] Received command c on object id p0
2026-04-27 08:22:28,051 [INFO] [DQ-Silver] silver_videos: 668 rows
2026-04-27 08:22:28,342 [INFO]   [not_null] video_id: ✓
2026-04-27 08:22:28,587 [INFO]   [positive] view_count: ✓
2026-04-27 08:22:28,824 [INFO]   [positive] like_count: ✓
2026-04-27 08:22:29,233 [INFO]   [positive] duration_seconds: ✓
2026-04-27 08:22:29,491 [INFO]   [not_null] publish_date: ✓
2026-04-27 08:22:30,065 [INFO] [DQ-Silver] silver_comments: 22,915 rows
2026-04-27 08:22:30,308 [INFO]   [not_null] comment_id: ✓
2026-04-27 08:22:30,553 [INFO]   [not_null] video_id: ✓
2026-04-27 08:22:30,808 [INFO]   [not_null] comment_text: ✓


## 7 · Summary

In [0]:
print("=" * 60)
print("  SILVER LAYER COMPLETE — SUMMARY")
print("=" * 60)
print(f"  Run Date              : {RUN_DATE}")
print(f"  silver_videos rows    : {sv_count:,}")
print(f"  silver_comments rows  : {sc_count:,}")
print(f"  Catalog               : {SILVER_DB}")
print(f"  Delta location        : {SILVER_PATH}")
print("=" * 60)

dbutils.notebook.exit(json.dumps({
    "status":                 "success",
    "run_date":               RUN_DATE,
    "silver_videos_count":    sv_count,
    "silver_comments_count":  sc_count,
}))